# 16 — TF-IDF Deep Dive

TF-IDF (term frequency × inverse document frequency) re-weights raw counts: a term is important if it appears often in *this* document but rarely in *other* documents. It is the most widely used text representation in classical ML and the default feature set for resume–JD matching.

**Why it matters for resumes / ATS:** raw counts rank "experience" and "data" as highly as "tensorflow" — nonsense for matching. TF-IDF demotes generic resume words and promotes distinctive skills, so the similarity between a resume and a job description is driven by what makes the resume specific, not by filler.

**Goal:** Understand TF-IDF weighting — the most popular text representation in classical ML.

This chapter goes from the math (TF and IDF by hand) to the library (`TfidfVectorizer`) to the application (resume–JD similarity). By the end you can explain why "java" scores 0.681 in one document while "data" scores 0.447 — and use that intuition to build a matcher.

## 1. TF-IDF Manually

TF is how often a word appears in a document (normalized by length); IDF is a corpus-level rarity penalty. The smoothed variant here is `log(N / (1 + df)) + 1`, which keeps IDF positive and dampens the zero-document case.

**What the code does:** defines `tf()` and `idf()` from scratch and scores the word "python" across three documents.
- Doc 1: `0.1250 × 1.0000 = 0.1250`; Doc 2: `0.1111 × 1.0000 = 0.1111`; Doc 3: `0.0000` (python absent).
- IDF is exactly `1.0000` everywhere because "python" appears in 2 of 3 documents: `log(3/(1+2)) + 1 = 1`. The +1 smoothing is why it never collapses to zero.

**Try it:** TF differs between docs (0.1250 vs 0.1111) because of the length normalization, while IDF is constant for the word across the corpus. That separation — local frequency vs global rarity — is the whole idea.

In [4]:
import numpy as np
from collections import Counter
import math
docs = [
    "Python is a programming language for data science",
    "Python is used in machine learning and data science",
    "Java is another programming language",
]
# Manual TF-IDF
def tf(word, doc):
    words = doc.lower().split()
    count = words.count(word.lower())
    return count / len(words) if len(words) > 0 else 0

def idf(word, docs):
    n_containing = sum(1 for d in docs if word.lower() in d.lower().split())
    return math.log(len(docs) / (1 + n_containing)) + 1

word = "python"
print(f"TF-IDF for '{word}':")
for i, d in enumerate(docs):
    tfidf = tf(word, d) * idf(word, docs)
    print(f"  Doc {i+1}: TF={tf(word,d):.4f} x IDF={idf(word,docs):.4f} = {tfidf:.4f}")

TF-IDF for 'python':
  Doc 1: TF=0.1250 x IDF=1.0000 = 0.1250
  Doc 2: TF=0.1111 x IDF=1.0000 = 0.1111
  Doc 3: TF=0.0000 x IDF=1.0000 = 0.0000


## 2. TF-IDF with scikit-learn

`TfidfVectorizer` applies the same idea with the library's normalization (L2 by default): each document vector is scaled to unit length, so vector comparisons measure term *mix* rather than document *length*.

**What the code does:** fits on the three docs with `stop_words="english", max_features=10` and prints each document's top-5 weighted terms.
- Vocabulary: `['data', 'java', 'language', 'learning', 'machine', 'programming', 'python', 'science', 'used']`.
- Doc 3's top term is `java` at `0.681` — it appears in only one document, so its IDF is maximal; shared words like `data` (`0.447`) rank lower.

**Try it:** compare doc 1 (five top terms tied at 0.447) with doc 3 (`java` 0.681, `language`/`programming` 0.518) — the document with a rare distinctive term has the sharpest weights.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(stop_words="english", max_features=10)
X = vec.fit_transform(docs)
print(f"Vocabulary: {vec.get_feature_names_out()}")
print("\nTF-IDF Matrix:")
for i in range(len(docs)):
    scores = [(w, X[i, j]) for w, j in zip(vec.get_feature_names_out(), range(X.shape[1])) if X[i, j] > 0]
    scores.sort(key=lambda x: -x[1])
    print(f"  Doc {i+1}: {[(w, round(s,3)) for w, s in scores[:5]]}")

Vocabulary: ['data' 'java' 'language' 'learning' 'machine' 'programming' 'python'
 'science' 'used']

TF-IDF Matrix:
  Doc 1: [('data', 0.447), ('language', 0.447), ('programming', 0.447), ('python', 0.447), ('science', 0.447)]
  Doc 2: [('learning', 0.46), ('machine', 0.46), ('used', 0.46), ('data', 0.349), ('python', 0.349)]
  Doc 3: [('java', 0.681), ('language', 0.518), ('programming', 0.518)]


## 3. TF-IDF for Resume-JD Matching

The canonical ATS operation: vectorize the resume and the job description into the *same* TF-IDF space, then measure their cosine similarity. Shared skills push the vectors together; generic terms barely matter.

**What the code does:** fits one `TfidfVectorizer` on `[resume, jd]`, prints the shared vocabulary, and computes cosine similarity between the two rows.
- Shared keywords include `python`, `tensorflow`, `machine`, `learning`, `nlp`, `data`, `science` — the actual overlap between candidate and job.
- `Resume-JD TF-IDF similarity: 0.617` — a solid match, driven by the tech-stack overlap.

**Try it:** swap one resume term for a synonym ("pytorch" instead of "tensorflow") and watch the score drop — TF-IDF has no synonym awareness. That limitation motivates the embedding chapters (Ch. 19+).

In [3]:
resume = "python tensorflow machine learning nlp data science experience"
jd = "python machine learning tensorflow deep learning data science required"
vec = TfidfVectorizer(stop_words="english")
X = vec.fit_transform([resume, jd])
features = vec.get_feature_names_out()
print(f"Shared keywords: {set(features)}")
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(X[0:1], X[1:2])[0][0]
print(f"Resume-JD TF-IDF similarity: {sim:.3f}")

Shared keywords: {'tensorflow', 'experience', 'required', 'learning', 'nlp', 'science', 'machine', 'deep', 'data', 'python'}
Resume-JD TF-IDF similarity: 0.617


## Key Insight: TF-IDF downweights common words ('experience', 'data') and highlights distinctive ones (skills, tech).

**TF-IDF's superpower is selectivity: it makes a document's identity come from its rare, meaningful terms.**

For resumes this is exactly right — "experience" tells you nothing, "tensorflow" tells you everything. The weighted vectors are what Ch. 18's cosine similarity consumes, and the ranking idea is what Ch. 14's keyword extraction used. Next, Ch. 17 adds word order back via n-grams, fixing TF-IDF's other blind spot.